In [ ]:
import yaml
import matplotlib.pyplot as plt
import numpy as np
import os
import tables_io
import pandas as pd

In [ ]:
RESULTS_DIR = '../results'
SUBMISSIONS = ['easy_forest', 'rail_knn_test', 'rail_bpz_test', 'bpz_31temps']

Y_LABEL_STRINGS = [
    'taskset_1_1yr_cardinal',
    'taskset_1_1yr_flagship',
    'taskset_1_10yr_cardinal',
    'taskset_1_10yr_flagship',
    'taskset_2_1yr_cardinal',
    'taskset_2_1yr_flagship',
    'taskset_2_10yr_cardinal',
    'taskset_2_10yr_flagship',
]

In [ ]:
data_dict = {}
for sub_ in SUBMISSIONS:
    submission_file = os.path.join(RESULTS_DIR, f"summary_{sub_}_point.parq")
    data_dict[sub_] = tables_io.read(submission_file)

In [ ]:
data_dict[sub_]

In [ ]:
def score_metrics(
    data,
    metric_dict,
    n_tasksets=2,
) -> int:
    score_dict = {k:np.zeros(n_tasksets, dtype=int) for k in metric_dict}
    norms = np.zeros(n_tasksets, dtype=int)
    taskset_scores = np.zeros(n_tasksets, dtype=int)
    for irow, row in data.iterrows():
        if row['task'] != 1:
            continue
        i_taskset = int(row['taskset'] - 1)
            
        for k, metric_ranges in metric_dict.items():
            norms[i_taskset] += len(metric_ranges)

            for metric_range in metric_ranges:
                if metric_range[0] <= row[k] and metric_range[1] >= row[k]:
                    score_dict[k][i_taskset] += 1
                    taskset_scores[i_taskset] += 1
    
    score_dict['norms'] = norms
    score_dict['taskset_scores'] = taskset_scores
    score_dict['percentages'] = taskset_scores/norms

    return score_dict
    
def score_all_metrics(
    data_dict,
    metric_dict,
):
    return {k:score_metrics(v, metric_dict) for k, v in data_dict.items()}


def extract_score(
    score_dict,
    which_score,
    n_tasksets=2,
):
    n_sub = len(score_dict)
    out_dict = dict(Submission=[])
    out_dict.update(**{f"Taskset_{i+1}":[] for i in range(n_tasksets)})
    for sub, scores in score_dict.items():
        out_dict['Submission'].append(sub)
        for i in range(n_tasksets):
            out_dict[f"Taskset_{i+1}"].append(np.round(scores[which_score][i], 2))
    return pd.DataFrame(out_dict)
    
    #return pd.DataFrame({k: v[which_score] for k, v in score_dict.items()})

In [ ]:
metric_dict = dict(
    mean=[[-0.01, 0.01], [-0.025, 0.025], [-0.05, 0.05]],
    std=[[0.0, 0.025], [0.0, 0.050], [0.0, 0.10]],
    abs_outlier_rate=[[0.0, 0.025], [0.0, 0.1], [0.0, 0.2]],
    CvM=[[0, 250], [0, 500], [0, 1000]],
    outlier=[[0, 0.2], [0, 0.4], [0, 0.6]],
    ks=[[0, 0.2], [0, 0.4], [0, 0.6]],
    ksamp=[[0, 1e3], [0, 2e3], [0, 4e3]],
)

In [ ]:
score_dict = score_all_metrics(data_dict, metric_dict)

In [ ]:
extract_score(score_dict, 'percentages')